# 🧬 RACIPE PROJECT - COMPLETE ANALYSIS

**Date:** 2026-09-02 17:03

---


## 1. 📦 Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print(" 🧬 RACIPE PROJECT - COMPLETE ANALYSIS")
print("="*70)

## 2. 📊 Load Data

In [ ]:
# RACIPE results
racipe_medians = pd.read_csv('racipe_10000_medians.csv')
racipe_scores = pd.read_csv('racipe_10000_emt_scores.csv')['emt_score']

# GSE69667 real data
real_data = pd.read_csv('GSE69667_EMT_scores_real.csv')

print(f"  ✅ RACIPE: {len(racipe_scores)} models")
print(f"  ✅ GSE69667: {len(real_data)} time points")
print(f"     Time points: {real_data['Time'].values}")

## 3. 📈 Descriptive Statistics

In [ ]:
e = sum(racipe_scores < 0.3)
h = sum((racipe_scores >= 0.3) & (racipe_scores <= 0.7))
m = sum(racipe_scores > 0.7)

print(f"\n📊 RACIPE State Distribution:")
print(f"  Epithelial:  {e/len(racipe_scores)*100:.1f}% ({e} models)")
print(f"  Hybrid:      {h/len(racipe_scores)*100:.1f}% ({h} models)")
print(f"  Mesenchymal: {m/len(racipe_scores)*100:.1f}% ({m} models)")

print(f"\n📊 RACIPE EMT Score:")
print(f"  Mean: {racipe_scores.mean():.3f} ± {racipe_scores.std():.3f}")
print(f"  Range: {racipe_scores.min():.3f} - {racipe_scores.max():.3f}")

print(f"\n📊 GSE69667 EMT Score:")
print(f"  Mean: {real_data['EMT_Score'].mean():.3f} ± {real_data['EMT_Score'].std():.3f}")
print(f"  Range: {real_data['EMT_Score'].min():.3f} - {real_data['EMT_Score'].max():.3f}")

## 4. 📈 Figures

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 4.1 EMT Score distribution
ax = axes[0, 0]
sns.kdeplot(racipe_scores, label='RACIPE (10k models)', ax=ax, color='blue', linewidth=2)
sns.kdeplot(real_data['EMT_Score'], label='GSE69667 (real)', ax=ax, color='red', linewidth=2)
ax.axvline(0.3, color='green', linestyle='--', alpha=0.7, label='E threshold')
ax.axvline(0.7, color='red', linestyle='--', alpha=0.7, label='M threshold')
ax.set_xlabel('EMT Score (N/(E+N))')
ax.set_ylabel('Density')
ax.set_title('RACIPE vs GSE69667')
ax.legend()
ax.grid(alpha=0.3)

# 4.2 Real trajectory
ax = axes[0, 1]
ax.plot(real_data['Time'], real_data['EMT_Score'], 'o-', color='red', linewidth=2, markersize=8)
ax.axhline(0.3, color='green', linestyle='--', alpha=0.7, label='E threshold')
ax.axhline(0.7, color='red', linestyle='--', alpha=0.7, label='M threshold')
ax.set_xlabel('Time (hours)')
ax.set_ylabel('EMT Score')
ax.set_title('GSE69667 Trajectory (A549 + TGF-β)')
ax.legend()
ax.grid(alpha=0.3)

# 4.3 Histogram
ax = axes[1, 0]
bins = np.linspace(0, 1, 50)
colors = {'Epithelial': 'green', 'Hybrid': 'orange', 'Mesenchymal': 'red'}
for name, mask in [('Epithelial', racipe_scores < 0.3),
                   ('Hybrid', (racipe_scores >= 0.3) & (racipe_scores <= 0.7)),
                   ('Mesenchymal', racipe_scores > 0.7)]:
    ax.hist(racipe_scores[mask], bins=bins, alpha=0.5, label=name, color=colors[name])
ax.axvline(0.3, color='black', linestyle='--', alpha=0.5)
ax.axvline(0.7, color='black', linestyle='--', alpha=0.5)
ax.set_xlabel('EMT Score')
ax.set_ylabel('Frequency')
ax.set_title('RACIPE State Distribution')
ax.legend()
ax.grid(alpha=0.3)

# 4.4 Boxplot
ax = axes[1, 1]
data_to_plot = [racipe_medians['E'], racipe_medians['N']]
bp = ax.boxplot(data_to_plot, labels=['E-cadherin', 'N-cadherin'], patch_artist=True)
for patch, color in zip(bp['boxes'], ['blue', 'red']):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)
ax.set_ylabel('Expression Level')
ax.set_title('E-cadherin and N-cadherin Distribution')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figure_final_summary.png', dpi=300)
print("  ✅ Saved: figure_final_summary.png")
plt.show()

## 5. 🔍 Correlation Analysis

In [ ]:
correlations = {}
for gene in racipe_medians.columns:
    corr, p = stats.spearmanr(racipe_medians[gene], racipe_scores)
    correlations[gene] = {'correlation': corr, 'p_value': p}

print("\n📊 Gene ranking by influence on EMT Score:")
sorted_genes = sorted(correlations.items(), key=lambda x: abs(x[1]['correlation']), reverse=True)
print(f"{'Gene':<12} {'Correlation':>15} {'p-value':>15}")
print("-"*45)
for gene, data in sorted_genes:
    print(f"{gene:<12} {data['correlation']:>15.3f} {data['p_value']:>15.6f}")

## 6. 🎯 PCA Analysis

In [ ]:
pca = PCA(n_components=2)
racipe_pca = pca.fit_transform(racipe_medians[['E', 'N']])
real_pca = pca.transform(real_data[['E-cadherin', 'N-cadherin']])

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(racipe_pca[:, 0], racipe_pca[:, 1], 
                     c=racipe_scores, cmap='RdYlGn_r', alpha=0.3, s=5)
ax.plot(real_pca[:, 0], real_pca[:, 1], 'o-', color='blue', linewidth=2, markersize=10)
for i, t in enumerate(real_data['Time']):
    ax.annotate(f'{t}h', (real_pca[i, 0], real_pca[i, 1]), fontsize=10)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('Real Trajectory in RACIPE Space')
plt.colorbar(scatter, label='EMT Score')
plt.tight_layout()
plt.savefig('figure_pca_trajectory.png', dpi=300)
print("  ✅ Saved: figure_pca_trajectory.png")
plt.show()

## 7. ✅ Summary

In [ ]:
print("\n" + "="*70)
print(" ✅ PROJECT COMPLETE!")
print("="*70)

print("\n📊 Final Summary:")
print(f"  1. RACIPE models: {len(racipe_scores)}")
print(f"  2. EMT Score range: {racipe_scores.min():.3f} - {racipe_scores.max():.3f}")
print(f"  3. State distribution: E={e/len(racipe_scores)*100:.1f}%, H={h/len(racipe_scores)*100:.1f}%, M={m/len(racipe_scores)*100:.1f}%")
print(f"  4. GSE69667 range: {real_data['EMT_Score'].min():.3f} - {real_data['EMT_Score'].max():.3f}")
print(f"  5. Most influential genes: {sorted_genes[0][0]} (r={sorted_genes[0][1]['correlation']:.3f}), {sorted_genes[1][0]} (r={sorted_genes[1][1]['correlation']:.3f})")

print("\n" + "="*70)
print("🎉 ALL DONE!")
print("="*70)